In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Kannada Handwritten Digit Recognition
#  Adapted from the Bengali (BanglaLekha-Isolated) version.
#
#  Transfer rationale
#  ──────────────────
#  Kannada is a South Indian Brahmic script.  Unlike Bengali/Devanagari it does
#  NOT have a shirorekha (top headline); instead characters are defined by
#  closed curves, loops, and prominent vertical/diagonal strokes.  Structural
#  changes vs. the Bengali version:
#
#    • num_classes   : 84  → 10   (Kannada-MNIST: digits 0–9 / ೦–೯)
#    • image_size    : 32  → 28   (Kannada-MNIST images are natively 28×28)
#    • Dataset format: folder-per-class (image files)
#                      → npz arrays  (Kannada-MNIST ships as .npz, identical
#                        layout to fashion-MNIST / MNIST)
#    • Dataset loader: Pillow file-walker → direct np.load() from .npz
#    • Scaffold branch: the 1×5 horizontal-headline conv is REPLACED by a
#                       1×3 + 3×1 cross-hair pair – better suited to the
#                       rounded, loop-heavy Kannada digit strokes.
#    • STM docstring  : updated for Kannada stroke characteristics
#    • Model name     : WhatNet-BanglaLekha → WhatNet-Kannada
#    • Output files   : banglalekha_*.json  → kannada_*.json
#
#  Everything else – architecture depth, augmentation, LR schedule,
#  callbacks, display utilities – is unchanged.
#
#  Dataset
#  ───────
#  Kannada-MNIST  (Prabhu, 2019)
#    • 60 000 train + 10 000 test 28×28 greyscale images of Kannada digits
#    • Hosted on Kaggle:
#        https://www.kaggle.com/datasets/higgstachyon/kannada-mnist
#    • Download produces:
#        kannada_mnist_train.npz  –  keys: arr_0 (images), arr_1 (labels)
#        kannada_mnist_test.npz   –  keys: arr_0 (images), arr_1 (labels)
#      OR the CSV variants:
#        train.csv  /  test.csv   –  label in column 0, pixels in columns 1-784
#    • This script handles BOTH .npz and .csv automatically.
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Fields marked  ← CHANGED  differ from the Bengali version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 10 classes – Kannada digits ೦ ೧ ೨ ೩ ೪ ೫ ೬ ೭ ೮ ೯
    "num_classes":      10,
    # ← CHANGED: Kannada-MNIST images are natively 28×28 (same as MNIST)
    "image_size":       28,
    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.05,
    "val_split":        0.1,

    # ← CHANGED: path to the Kannada-MNIST directory on Kaggle.
    #   The folder should contain one or more of:
    #     kannada_mnist_train.npz  +  kannada_mnist_test.npz   ← preferred
    #     train.csv                +  test.csv                  ← fallback
    #
    #   Kaggle path (after adding the dataset to your notebook):
    #     "/kaggle/input/kannada-mnist"
    #
    #   Local path example:
    #     "/home/user/data/kannada-mnist"
    "data_dir":    "/kaggle/input/competitions/Kannada-MNIST",            # ← CHANGED

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     ← CHANGED entirely: Kannada-MNIST is distributed as .npz / .csv arrays,
#     not as a folder-per-class image tree.  We load arrays directly with
#     numpy, then wrap them in tf.data.Dataset – no Pillow walker needed.
# ─────────────────────────────────────────────────────────────────────────────
if not os.path.exists(CFG["data_dir"]):
    raise FileNotFoundError(
        f"[ERROR] Dataset directory not found: {CFG['data_dir']}\n"
        "Download Kannada-MNIST from Kaggle:\n"
        "  https://www.kaggle.com/datasets/higgstachyon/kannada-mnist\n"
        "and place the .npz (or .csv) files in that directory."
    )

def _load_npz(path: str):
    """Load images + labels from a Kannada-MNIST .npz file."""
    data   = np.load(path)
    images = data["arr_0"].astype(np.float32)   # shape (N, 28, 28)
    labels = data["arr_1"].astype(np.int32)      # shape (N,)
    return images, labels

def _load_csv(path: str):
    """
    Load images + labels from a Kannada-MNIST CSV file.
    Column 0 = label, columns 1-784 = flattened pixel values (0-255).
    """
    raw    = np.loadtxt(path, delimiter=",", skiprows=1, dtype=np.float32)
    labels = raw[:, 0].astype(np.int32)
    images = raw[:, 1:].reshape(-1, IMG, IMG).astype(np.float32)
    return images, labels

def _load_split(split: str):
    """
    Try .npz first, fall back to .csv.
    split must be 'train' or 'test'.
    """
    npz_path = os.path.join(CFG["data_dir"], f"kannada_mnist_{split}.npz")
    csv_path = os.path.join(CFG["data_dir"], f"{split}.csv")
    if os.path.exists(npz_path):
        print(f"[INFO] Loading {split} split from {npz_path}")
        return _load_npz(npz_path)
    elif os.path.exists(csv_path):
        print(f"[INFO] Loading {split} split from {csv_path}")
        return _load_csv(csv_path)
    else:
        raise FileNotFoundError(
            f"[ERROR] Could not find {split} data.\n"
            f"  Looked for: {npz_path}\n"
            f"         and: {csv_path}\n"
            "Please verify the Kannada-MNIST files are in CFG['data_dir']."
        )

# ── Load raw arrays ───────────────────────────────────────────────────────────
x_train_all, y_train_all = _load_split("train")
x_test,      y_test      = _load_split("test")

print(f"[INFO] Classes in dataset : {np.unique(y_train_all).tolist()}")
print(f"[INFO] CFG num_classes    : {NUM_CLASSES}")
if len(np.unique(y_train_all)) != NUM_CLASSES:
    print(f"[WARN] Mismatch – update CFG['num_classes'] to "
          f"{len(np.unique(y_train_all))} to match your dataset.")

# ── Train / val split ─────────────────────────────────────────────────────────
n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

# Shuffle before splitting (dataset is class-sorted in the .npz)
rng  = np.random.default_rng(SEED)
perm = rng.permutation(n_total)
x_train_all, y_train_all = x_train_all[perm], y_train_all[perm]

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

# ── Add channel dim: (N,28,28) → (N,28,28,1) ─────────────────────────────────
x_train = x_train[..., np.newaxis]
x_val   = x_val[...,   np.newaxis]
x_test  = x_test[...,  np.newaxis]

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Light stochastic augmentation – training only.
    Kannada digit glyphs are compact and mostly upright, so we keep
    augmentations mild: brightness/contrast jitter + pad-and-random-crop
    for random translation.  No rotation – it would corrupt loop topology.
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
#    Images are already in RAM as numpy arrays, so from_tensor_slices is
#    ideal – no file I/O or Pillow decoding required.
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
# test_ds      – raw integer labels  (used by compute_macro_f1)
# test_ds_oh   – one-hot labels      (used by model.evaluate)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged from Bengali version)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    """GELU activation – smoother than ReLU, better gradients in deep nets."""
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    """Standard pre-activation residual block."""
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """
    DenseNet-inspired residual block.
    Three sequential residual blocks with dense connections, then a
    1×1 bottleneck and stride-2 depthwise downsampling.
    """
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    """Squeeze-and-Excitation (SE) channel attention."""
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    """Lightweight capsule-like routing module. O(n) cost, no dynamic routing."""
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Kannada stroke topology encoder.                          ← CHANGED docstring

    Kannada digits (೦–೯) are characterised by closed loops,
    rounded curves, and mixed horizontal/vertical strokes – very
    different from the shirorekha-dominated Bengali/Devanagari scripts.

    Kernel choices:
      • 1×3 branch  – short horizontal strokes and top/bottom curves
      • 3×1 branch  – vertical strokes (e.g. the spine in ೧, ೭)
      • dilation=1  – local curve detail
      • dilation=2  – semi-local loop context (captures the full extent
                       of rounded glyphs at 28-px resolution)

    The 1×5 headline-specific conv used for Bengali/Devanagari is replaced
    by the shorter 1×3 because Kannada digits lack a continuous headline.
    """
    # ← CHANGED: 1×5 → 1×3  (no shirorekha in Kannada)
    h  = layers.Conv2D(64, (1, 3), padding="same", use_bias=False, activation=gelu)(x)
    # ← CHANGED: 5×1 → 3×1  (symmetric with horizontal branch)
    v  = layers.Conv2D(64, (3, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    out = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    """
    Cross-scale transformer bridge (CSTB).
    Pools three encoder-scale feature maps to tokens and applies multi-head
    self-attention so fine-grained and coarse features interact.
    """
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     ← CHANGED: renamed to WhatNet-Kannada; num_classes=10; image_size=28.
#     All architectural decisions carry over unchanged.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_kannada(num_classes: int = 10, image_size: int = 28) -> Model:
    """
    WhatNet-Kannada: WhatNet adapted for Kannada-MNIST digit recognition.

    Architecture overview
    ─────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (texture / overall shape)
      • Cross-hair scaffold: 1×3 conv (horizontal) – captures short
        horizontal strokes and top/bottom curve segments of Kannada digits
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (14×14 at 28-px input)
      enc2: 64→128   ( 7× 7)
      enc3: 128→256  ( 4× 4, with padding)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – 1×3 + 3×1 + dilated branches
      • Gated fusion → final MLP + layer norm → logits (10 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    # ← CHANGED scaffold: 1×5 (Bengali headline) → 1×3 (Kannada horizontal strokes)
    s        = layers.Conv2D(32, (1, 3), padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(7)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    # ← CHANGED pool factor: 8 → 7  (28/4 = 7, not 4; keeps spatial dims aligned)
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    afc_in   = layers.GlobalAveragePooling2D()(enc3)
    afc_in   = layers.Add()([afc_in, cstb_out])
    afc_out  = adaptive_filter_capsule(afc_in, K)

    stgm_out    = stroke_topology_module(enc3, 256)
    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Kannada")  # ← CHANGED

MODELS_TF = {
    "WhatNet-Kannada": lambda: build_whatnet_kannada(NUM_CLASSES, IMG),  # ← CHANGED
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = -(-n_train // BS)   # ceiling division
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs  "
         f"[Kannada-MNIST]", "bold"))                                   # ← CHANGED
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "kannada_results.json")   # ← CHANGED
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kannada_histories.json")  # ← CHANGED
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kannada-MNIST benchmark complete.\n", "green", "bold"))  # ← CHANGED

2026-04-19 02:41:06.838002: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776566467.030868      56 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776566467.093314      56 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776566467.563868      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776566467.563915      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776566467.563920      56 computation_placer.cc:177] computation placer alr

[INFO] Loading train split from /kaggle/input/competitions/Kannada-MNIST/train.csv
[INFO] Loading test split from /kaggle/input/competitions/Kannada-MNIST/test.csv
[INFO] Classes in dataset : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[INFO] CFG num_classes    : 10
[INFO] Train: 54,000 | Val: 6,000 | Test: 5,000


I0000 00:00:1776566496.008771      56 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776566496.014692      56 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 50 epochs  [Kannada-MNIST]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Kannada  –  Parameter Summary                       ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d          ║  Conv2D               ║              288  ║
║  conv2d_1        ║  Conv2D               ║               96  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  dense           ║  Dense                ║              520  ║
║  dense_1         ║  Dense                ║              576  ║
║  conv2d_2        ║  Conv2D               ║            4,096  ║
║  batch_normal

I0000 00:00:1776566523.647119     124 service.cc:152] XLA service 0x78aabc01a630 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776566523.647166     124 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776566523.647173     124 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776566528.346117     124 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776566551.161810     124 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Epoch   1/50  ░░░░░░░░░░░░░░░░░░░░   2.0%  loss=0.3903  acc=96.15%  val_loss=0.3594  val_acc=97.60%  lr=5.00e-04  [152.9s]
  Epoch   2/50  ░░░░░░░░░░░░░░░░░░░░   4.0%  loss=0.3109  acc=99.11%  val_loss=0.3340  val_acc=98.20%  lr=4.98e-04  [70.1s]
  Epoch   3/50  █░░░░░░░░░░░░░░░░░░░   6.0%  loss=0.3064  acc=99.23%  val_loss=0.3237  val_acc=98.87%  lr=4.96e-04  [69.4s]
  Epoch   4/50  █░░░░░░░░░░░░░░░░░░░   8.0%  loss=0.3027  acc=99.33%  val_loss=0.3032  val_acc=99.13%  lr=4.92e-04  [69.3s]
  Epoch   5/50  ██░░░░░░░░░░░░░░░░░░  10.0%  loss=0.3020  acc=99.42%  val_loss=0.3187  val_acc=98.77%  lr=4.88e-04  [68.9s]
  Epoch   6/50  ██░░░░░░░░░░░░░░░░░░  12.0%  loss=0.2993  acc=99.48%  val_loss=0.3074  val_acc=99.08%  lr=4.82e-04  [68.8s]
  Epoch   7/50  ██░░░░░░░░░░░░░░░░░░  14.0%  loss=0.2990  acc=99.45%  val_loss=0.3076  val_acc=99.12%  lr=4.76e-04  [68.6s]
  Epoch   8/50  ███░░░░░░░░░░░░░░░░░  16.0%  loss=0.2969  acc=99.51%  val_loss=0.3095  val_acc=99.08%  lr=4.69e-04  [68.8s]
  Epoch

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Kannada Handwritten Digit Recognition
#  Adapted from the Bengali (BanglaLekha-Isolated) version.
#
#  Transfer rationale
#  ──────────────────
#  Kannada is a South Indian Brahmic script.  Unlike Bengali/Devanagari it does
#  NOT have a shirorekha (top headline); instead characters are defined by
#  closed curves, loops, and prominent vertical/diagonal strokes.  Structural
#  changes vs. the Bengali version:
#
#    • num_classes   : 84  → 10   (Kannada-MNIST: digits 0–9 / ೦–೯)
#    • image_size    : 32  → 28   (Kannada-MNIST images are natively 28×28)
#    • Dataset format: folder-per-class (image files)
#                      → npz arrays  (Kannada-MNIST ships as .npz, identical
#                        layout to fashion-MNIST / MNIST)
#    • Dataset loader: Pillow file-walker → direct np.load() from .npz
#    • Scaffold branch: the 1×5 horizontal-headline conv is REPLACED by a
#                       1×3 + 3×1 cross-hair pair – better suited to the
#                       rounded, loop-heavy Kannada digit strokes.
#    • STM docstring  : updated for Kannada stroke characteristics
#    • Model name     : WhatNet-BanglaLekha → WhatNet-Kannada
#    • Output files   : banglalekha_*.json  → kannada_*.json
#
#  Everything else – architecture depth, augmentation, LR schedule,
#  callbacks, display utilities – is unchanged.
#
#  Dataset
#  ───────
#  Kannada-MNIST  (Prabhu, 2019)
#    • 60 000 train + 10 000 test 28×28 greyscale images of Kannada digits
#    • Hosted on Kaggle:
#        https://www.kaggle.com/datasets/higgstachyon/kannada-mnist
#    • Download produces:
#        kannada_mnist_train.npz  –  keys: arr_0 (images), arr_1 (labels)
#        kannada_mnist_test.npz   –  keys: arr_0 (images), arr_1 (labels)
#      OR the CSV variants:
#        train.csv  /  test.csv   –  label in column 0, pixels in columns 1-784
#    • This script handles BOTH .npz and .csv automatically.
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Fields marked  ← CHANGED  differ from the Bengali version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 10 classes – Kannada digits ೦ ೧ ೨ ೩ ೪ ೫ ೬ ೭ ೮ ೯
    "num_classes":      10,

    # ← CHANGED: Kannada-MNIST images are natively 28×28 (same as MNIST)
    "image_size":       28,

    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.05,
    "val_split":        0.1,

    # ← CHANGED: path to the Kannada-MNIST directory on Kaggle.
    #
    #   ⚠️  IMPORTANT – use the DATASET path, NOT the competition path:
    #
    #   ✅ CORRECT  (higgstachyon's dataset – has real test labels):
    #        "/kaggle/input/kannada-mnist"
    #        Files: kannada_mnist_train.npz  ← 60 000 labelled train images
    #               kannada_mnist_test.npz   ← 10 000 labelled test images
    #
    #   ❌ WRONG  (competition folder – test labels are DUMMY zeros):
    #        "/kaggle/input/competitions/Kannada-MNIST"
    #        Files: train.csv  (60 000 rows, labelled – ok for training)
    #               test.csv   (5 000 rows, label col = 0 always – NOT usable
    #                           for evaluation; this is the submission file)
    #
    #   If you only have the competition folder, you MUST either:
    #     a) Add the higgstachyon dataset to your notebook (preferred), OR
    #     b) Use the val split for final reporting (see NOTE below).
    #
    #   The script raises a clear error if it detects the unlabelled file.
    "data_dir":    "/kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST",            # ← CORRECT path

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     ← CHANGED entirely: Kannada-MNIST is distributed as .npz / .csv arrays,
#     not as a folder-per-class image tree.  We load arrays directly with
#     numpy, then wrap them in tf.data.Dataset – no Pillow walker needed.
# ─────────────────────────────────────────────────────────────────────────────
if not os.path.exists(CFG["data_dir"]):
    raise FileNotFoundError(
        f"[ERROR] Dataset directory not found: {CFG['data_dir']}\n"
        "Download Kannada-MNIST from Kaggle:\n"
        "  https://www.kaggle.com/datasets/higgstachyon/kannada-mnist\n"
        "and place the .npz (or .csv) files in that directory."
    )

def _load_npz(path: str):
    """Load images + labels from a Kannada-MNIST .npz file."""
    data   = np.load(path)
    images = data["arr_0"].astype(np.float32)   # shape (N, 28, 28)
    labels = data["arr_1"].astype(np.int32)      # shape (N,)
    return images, labels

def _load_csv(path: str, split: str):
    """
    Load images + labels from a Kannada-MNIST CSV file.
    Column 0 = label, columns 1-784 = flattened pixel values (0-255).

    GUARD: The Kaggle competition test.csv has only 5 000 rows and its label
    column is always 0 (a dummy placeholder for the submission format).
    Evaluating against it produces a score near 10% (random chance for 10
    classes) even for a well-trained model.  We detect this and raise a clear
    error so the user is not misled.
    """
    import csv
    with open(path, newline="") as f:
        header = next(csv.reader(f))

    # Competition test.csv header is typically: id,pixel0,pixel1,...
    # Dataset test.csv / train.csv header is:   label,pixel0,pixel1,...
    if split == "test" and header[0].strip().lower() != "label":
        raise ValueError(
            "[ERROR] The test CSV looks like the Kaggle COMPETITION file "
            f"(first column is '{header[0]}', expected 'label').\n"
            "This file has dummy/zero labels and CANNOT be used for evaluation.\n"
            "Fix: add the higgstachyon dataset to your notebook and set\n"
            "  CFG['data_dir'] = '/kaggle/input/kannada-mnist'\n"
            "which contains kannada_mnist_test.npz with 10 000 real labels.\n"
            "Alternatively, report performance on the validation split instead."
        )

    raw    = np.loadtxt(path, delimiter=",", skiprows=1, dtype=np.float32)
    labels = raw[:, 0].astype(np.int32)
    images = raw[:, 1:].reshape(-1, IMG, IMG).astype(np.float32)

    # Secondary guard: if all test labels are 0, something is wrong.
    if split == "test" and np.all(labels == 0):
        raise ValueError(
            "[ERROR] All labels in the test CSV are 0 – this is the unlabelled\n"
            "competition submission file, not the evaluated test set.\n"
            "Set CFG['data_dir'] to the higgstachyon dataset path:\n"
            "  '/kaggle/input/kannada-mnist'  (contains real test labels)."
        )

    return images, labels

def _load_split(split: str):
    """
    Try .npz first (preferred – always has real labels), fall back to .csv.
    split must be 'train' or 'test'.

    NOTE on the two Kaggle paths for this dataset:
      ✅ /kaggle/input/kannada-mnist          ← higgstachyon dataset
           kannada_mnist_train.npz  60 000 labelled samples
           kannada_mnist_test.npz   10 000 labelled samples   ← CORRECT test
      ❌ /kaggle/input/competitions/Kannada-MNIST  ← competition folder
           train.csv  60 000 rows (labels valid)
           test.csv    5 000 rows (label col = 0, DUMMY – submission format)
    """
    npz_path = os.path.join(CFG["data_dir"], f"X_kannada_MNIST_{split}.npz")
    csv_path = os.path.join(CFG["data_dir"], f"{split}.csv")
    if os.path.exists(npz_path):
        print(f"[INFO] Loading {split} split from {npz_path}")
        return _load_npz(npz_path)
    elif os.path.exists(csv_path):
        print(f"[INFO] Loading {split} split from {csv_path}")
        return _load_csv(csv_path, split)   # passes split for the label guard
    else:
        raise FileNotFoundError(
            f"[ERROR] Could not find {split} data.\n"
            f"  Looked for .npz : {npz_path}\n"
            f"  Looked for .csv : {csv_path}\n"
            "Recommended fix: add the higgstachyon Kaggle dataset and set\n"
            "  CFG['data_dir'] = '/kaggle/input/kannada-mnist'"
        )

# ── Load raw arrays ───────────────────────────────────────────────────────────
x_train_all, y_train_all = _load_split("train")
x_test,      y_test      = _load_split("test")

# Sanity-check: the real Kannada-MNIST test set has exactly 10 000 samples.
# The competition submission test.csv has 5 000 – catch it here as a last resort.
if len(x_test) == 5000:
    raise ValueError(
        "[ERROR] Test set has 5 000 samples – this matches the Kaggle COMPETITION\n"
        "submission file (test.csv), which has dummy labels and cannot be used\n"
        "for evaluation. The real dataset test split has 10 000 labelled samples.\n"
        "\n"
        "Fix: in your Kaggle notebook, click '+ Add Data' and add:\n"
        "  'Kannada-MNIST' by higgstachyon\n"
        "  https://www.kaggle.com/datasets/higgstachyon/kannada-mnist\n"
        "Then set:\n"
        "  CFG['data_dir'] = '/kaggle/input/kannada-mnist'\n"
        "\n"
        "If you only have the competition folder and cannot add the dataset,\n"
        "comment out the x_test / y_test lines and report on val split:\n"
        "  test_ds    = val_ds\n"
        "  test_ds_oh = val_ds  (already has one-hot labels)"
    )

print(f"[INFO] Classes in dataset : {np.unique(y_train_all).tolist()}")
print(f"[INFO] CFG num_classes    : {NUM_CLASSES}")
if len(np.unique(y_train_all)) != NUM_CLASSES:
    print(f"[WARN] Mismatch – update CFG['num_classes'] to "
          f"{len(np.unique(y_train_all))} to match your dataset.")

# ── Train / val split ─────────────────────────────────────────────────────────
n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

# Shuffle before splitting (dataset is class-sorted in the .npz)
rng  = np.random.default_rng(SEED)
perm = rng.permutation(n_total)
x_train_all, y_train_all = x_train_all[perm], y_train_all[perm]

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

# ── Add channel dim: (N,28,28) → (N,28,28,1) ─────────────────────────────────
x_train = x_train[..., np.newaxis]
x_val   = x_val[...,   np.newaxis]
x_test  = x_test[...,  np.newaxis]

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Light stochastic augmentation – training only.
    Kannada digit glyphs are compact and mostly upright, so we keep
    augmentations mild: brightness/contrast jitter + pad-and-random-crop
    for random translation.  No rotation – it would corrupt loop topology.
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
#    Images are already in RAM as numpy arrays, so from_tensor_slices is
#    ideal – no file I/O or Pillow decoding required.
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
# test_ds      – raw integer labels  (used by compute_macro_f1)
# test_ds_oh   – one-hot labels      (used by model.evaluate)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged from Bengali version)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    """GELU activation – smoother than ReLU, better gradients in deep nets."""
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    """Standard pre-activation residual block."""
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """
    DenseNet-inspired residual block.
    Three sequential residual blocks with dense connections, then a
    1×1 bottleneck and stride-2 depthwise downsampling.
    """
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    """Squeeze-and-Excitation (SE) channel attention."""
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    """Lightweight capsule-like routing module. O(n) cost, no dynamic routing."""
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Kannada stroke topology encoder.                          ← CHANGED docstring

    Kannada digits (೦–೯) are characterised by closed loops,
    rounded curves, and mixed horizontal/vertical strokes – very
    different from the shirorekha-dominated Bengali/Devanagari scripts.

    Kernel choices:
      • 1×3 branch  – short horizontal strokes and top/bottom curves
      • 3×1 branch  – vertical strokes (e.g. the spine in ೧, ೭)
      • dilation=1  – local curve detail
      • dilation=2  – semi-local loop context (captures the full extent
                       of rounded glyphs at 28-px resolution)

    The 1×5 headline-specific conv used for Bengali/Devanagari is replaced
    by the shorter 1×3 because Kannada digits lack a continuous headline.
    """
    # ← CHANGED: 1×5 → 1×3  (no shirorekha in Kannada)
    h  = layers.Conv2D(64, (1, 3), padding="same", use_bias=False, activation=gelu)(x)
    # ← CHANGED: 5×1 → 3×1  (symmetric with horizontal branch)
    v  = layers.Conv2D(64, (3, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    out = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    """
    Cross-scale transformer bridge (CSTB).
    Pools three encoder-scale feature maps to tokens and applies multi-head
    self-attention so fine-grained and coarse features interact.
    """
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     ← CHANGED: renamed to WhatNet-Kannada; num_classes=10; image_size=28.
#     All architectural decisions carry over unchanged.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_kannada(num_classes: int = 10, image_size: int = 28) -> Model:
    """
    WhatNet-Kannada: WhatNet adapted for Kannada-MNIST digit recognition.

    Architecture overview
    ─────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (texture / overall shape)
      • Cross-hair scaffold: 1×3 conv (horizontal) – captures short
        horizontal strokes and top/bottom curve segments of Kannada digits
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (14×14 at 28-px input)
      enc2: 64→128   ( 7× 7)
      enc3: 128→256  ( 4× 4, with padding)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – 1×3 + 3×1 + dilated branches
      • Gated fusion → final MLP + layer norm → logits (10 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    # ← CHANGED scaffold: 1×5 (Bengali headline) → 1×3 (Kannada horizontal strokes)
    s        = layers.Conv2D(32, (1, 3), padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(7)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    # ← CHANGED pool factor: 8 → 7  (28/4 = 7, not 4; keeps spatial dims aligned)
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    afc_in   = layers.GlobalAveragePooling2D()(enc3)
    afc_in   = layers.Add()([afc_in, cstb_out])
    afc_out  = adaptive_filter_capsule(afc_in, K)

    stgm_out    = stroke_topology_module(enc3, 256)
    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Kannada")  # ← CHANGED

MODELS_TF = {
    "WhatNet-Kannada": lambda: build_whatnet_kannada(NUM_CLASSES, IMG),  # ← CHANGED
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = -(-n_train // BS)   # ceiling division
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs  "
         f"[Kannada-MNIST]", "bold"))                                   # ← CHANGED
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "kannada_results.json")   # ← CHANGED
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kannada_histories.json")  # ← CHANGED
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kannada-MNIST benchmark complete.\n", "green", "bold"))  # ← CHANGED

FileNotFoundError: [ERROR] Could not find train data.
  Looked for .npz : /kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST/Kannada_MNIST_train.npz
  Looked for .csv : /kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST/train.csv
Recommended fix: add the higgstachyon Kaggle dataset and set
  CFG['data_dir'] = '/kaggle/input/kannada-mnist'

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Kannada Handwritten Digit Recognition
#  Adapted from the Bengali (BanglaLekha-Isolated) version.
#
#  Transfer rationale
#  ──────────────────
#  Kannada is a South Indian Brahmic script.  Unlike Bengali/Devanagari it does
#  NOT have a shirorekha (top headline); instead characters are defined by
#  closed curves, loops, and prominent vertical/diagonal strokes.  Structural
#  changes vs. the Bengali version:
#
#    • num_classes   : 84  → 10   (Kannada-MNIST: digits 0–9 / ೦–೯)
#    • image_size    : 32  → 28   (Kannada-MNIST images are natively 28×28)
#    • Dataset format: folder-per-class (image files)
#                      → npz arrays  (Kannada-MNIST ships as .npz, identical
#                        layout to fashion-MNIST / MNIST)
#    • Dataset loader: Pillow file-walker → direct np.load() from .npz
#    • Scaffold branch: the 1×5 horizontal-headline conv is REPLACED by a
#                       1×3 + 3×1 cross-hair pair – better suited to the
#                       rounded, loop-heavy Kannada digit strokes.
#    • STM docstring  : updated for Kannada stroke characteristics
#    • Model name     : WhatNet-BanglaLekha → WhatNet-Kannada
#    • Output files   : banglalekha_*.json  → kannada_*.json
#
#  Everything else – architecture depth, augmentation, LR schedule,
#  callbacks, display utilities – is unchanged.
#
#  Dataset
#  ───────
#  Kannada-MNIST  (Prabhu, 2019)
#    • 60 000 train + 10 000 test 28×28 greyscale images of Kannada digits
#    • Hosted on Kaggle:
#        https://www.kaggle.com/datasets/higgstachyon/kannada-mnist
#    • Download produces:
#        kannada_mnist_train.npz  –  keys: arr_0 (images), arr_1 (labels)
#        kannada_mnist_test.npz   –  keys: arr_0 (images), arr_1 (labels)
#      OR the CSV variants:
#        train.csv  /  test.csv   –  label in column 0, pixels in columns 1-784
#    • This script handles BOTH .npz and .csv automatically.
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Fields marked  ← CHANGED  differ from the Bengali version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 10 classes – Kannada digits ೦ ೧ ೨ ೩ ೪ ೫ ೬ ೭ ೮ ೯
    "num_classes":      10,

    # ← CHANGED: Kannada-MNIST images are natively 28×28 (same as MNIST)
    "image_size":       28,

    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.05,
    "val_split":        0.1,

    # Path to the Kannada-MNIST dataset root on Kaggle.
    #
    #   After adding the dataset to your notebook, the directory tree is:
    #
    #     /kaggle/input/kannada-mnist/
    #       Kannada_MNIST_npz/
    #         Dig_MNIST/          ← Dig-MNIST subset (not used here)
    #         Kannada_MNIST/      ← main split used by this script
    #           X_kannada_MNIST_train.npz   images  (60 000 × 28 × 28, uint8)
    #           y_kannada_MNIST_train.npz   labels  (60 000,           uint8)
    #           X_kannada_MNIST_test.npz    images  (10 000 × 28 × 28, uint8)
    #           y_kannada_MNIST_test.npz    labels  (10 000,           uint8)
    #
    #   Each .npz file contains a single array under the key 'arr_0'.
    #   Images are uint8 [0-255]; labels are uint8 [0-9].
    #
    #   Kaggle dataset page:
    #     https://www.kaggle.com/datasets/higgstachyon/kannada-mnist
    "data_dir":    "/kaggle/input/datasets/rautranjit/kannada-mnist",

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     Kannada-MNIST ships as split npz files (separate X / y per split) inside
#     a subdirectory.  Each .npz has a single array under key 'arr_0'.
#     Layout:
#       <data_dir>/Kannada_MNIST_npz/Kannada_MNIST/
#           X_kannada_MNIST_train.npz   y_kannada_MNIST_train.npz
#           X_kannada_MNIST_test.npz    y_kannada_MNIST_test.npz
# ─────────────────────────────────────────────────────────────────────────────

# Resolved subdirectory that holds the four .npz files
_NPZ_DIR = os.path.join(
    CFG["data_dir"], "Kannada_MNIST_npz", "Kannada_MNIST"
)

if not os.path.exists(CFG["data_dir"]):
    raise FileNotFoundError(
        f"[ERROR] Dataset root not found: {CFG['data_dir']}\n"
        "Add the dataset to your Kaggle notebook:\n"
        "  https://www.kaggle.com/datasets/higgstachyon/kannada-mnist\n"
        "Expected layout after mounting:\n"
        "  <data_dir>/Kannada_MNIST_npz/Kannada_MNIST/\n"
        "      X_kannada_MNIST_train.npz\n"
        "      y_kannada_MNIST_train.npz\n"
        "      X_kannada_MNIST_test.npz\n"
        "      y_kannada_MNIST_test.npz"
    )

if not os.path.exists(_NPZ_DIR):
    raise FileNotFoundError(
        f"[ERROR] Expected subdirectory not found: {_NPZ_DIR}\n"
        "The dataset appears to be mounted at the right root but the internal\n"
        "folder structure doesn't match.  Check that the dataset version on\n"
        "Kaggle still uses the path:\n"
        "  Kannada_MNIST_npz/Kannada_MNIST/<X|y>_kannada_MNIST_<train|test>.npz"
    )

def _load_split(split: str):
    """
    Load images and labels for one split from the two separate .npz files.
    Each file contains a single array under the key 'arr_0'.

    split: 'train'  →  60 000 samples (uint8 images, uint8 labels 0-9)
           'test'   →  10 000 samples (uint8 images, uint8 labels 0-9)
    """
    x_path = os.path.join(_NPZ_DIR, f"X_kannada_MNIST_{split}.npz")
    y_path = os.path.join(_NPZ_DIR, f"y_kannada_MNIST_{split}.npz")

    for p in (x_path, y_path):
        if not os.path.exists(p):
            raise FileNotFoundError(
                f"[ERROR] Missing file: {p}\n"
                f"Expected both X_kannada_MNIST_{split}.npz and "
                f"y_kannada_MNIST_{split}.npz in:\n  {_NPZ_DIR}"
            )

    print(f"[INFO] Loading {split} images from {x_path}")
    print(f"[INFO] Loading {split} labels from {y_path}")

    images = np.load(x_path)["arr_0"].astype(np.float32)  # (N, 28, 28)
    labels = np.load(y_path)["arr_0"].astype(np.int32)    # (N,)
    return images, labels

# ── Load raw arrays ───────────────────────────────────────────────────────────
x_train_all, y_train_all = _load_split("train")
x_test,      y_test      = _load_split("test")

print(f"[INFO] Classes in dataset : {np.unique(y_train_all).tolist()}")
print(f"[INFO] CFG num_classes    : {NUM_CLASSES}")
if len(np.unique(y_train_all)) != NUM_CLASSES:
    print(f"[WARN] Mismatch – update CFG['num_classes'] to "
          f"{len(np.unique(y_train_all))} to match your dataset.")

# ── Train / val split ─────────────────────────────────────────────────────────
n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

# Shuffle before splitting (dataset is class-sorted in the .npz)
rng  = np.random.default_rng(SEED)
perm = rng.permutation(n_total)
x_train_all, y_train_all = x_train_all[perm], y_train_all[perm]

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

# ── Add channel dim: (N,28,28) → (N,28,28,1) ─────────────────────────────────
x_train = x_train[..., np.newaxis]
x_val   = x_val[...,   np.newaxis]
x_test  = x_test[...,  np.newaxis]

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Light stochastic augmentation – training only.
    Kannada digit glyphs are compact and mostly upright, so we keep
    augmentations mild: brightness/contrast jitter + pad-and-random-crop
    for random translation.  No rotation – it would corrupt loop topology.
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
#    Images are already in RAM as numpy arrays, so from_tensor_slices is
#    ideal – no file I/O or Pillow decoding required.
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
# test_ds      – raw integer labels  (used by compute_macro_f1)
# test_ds_oh   – one-hot labels      (used by model.evaluate)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged from Bengali version)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    """GELU activation – smoother than ReLU, better gradients in deep nets."""
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    """Standard pre-activation residual block."""
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """
    DenseNet-inspired residual block.
    Three sequential residual blocks with dense connections, then a
    1×1 bottleneck and stride-2 depthwise downsampling.
    """
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    """Squeeze-and-Excitation (SE) channel attention."""
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    """Lightweight capsule-like routing module. O(n) cost, no dynamic routing."""
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Kannada stroke topology encoder.                          ← CHANGED docstring

    Kannada digits (೦–೯) are characterised by closed loops,
    rounded curves, and mixed horizontal/vertical strokes – very
    different from the shirorekha-dominated Bengali/Devanagari scripts.

    Kernel choices:
      • 1×3 branch  – short horizontal strokes and top/bottom curves
      • 3×1 branch  – vertical strokes (e.g. the spine in ೧, ೭)
      • dilation=1  – local curve detail
      • dilation=2  – semi-local loop context (captures the full extent
                       of rounded glyphs at 28-px resolution)

    The 1×5 headline-specific conv used for Bengali/Devanagari is replaced
    by the shorter 1×3 because Kannada digits lack a continuous headline.
    """
    # ← CHANGED: 1×5 → 1×3  (no shirorekha in Kannada)
    h  = layers.Conv2D(64, (1, 3), padding="same", use_bias=False, activation=gelu)(x)
    # ← CHANGED: 5×1 → 3×1  (symmetric with horizontal branch)
    v  = layers.Conv2D(64, (3, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    out = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    """
    Cross-scale transformer bridge (CSTB).
    Pools three encoder-scale feature maps to tokens and applies multi-head
    self-attention so fine-grained and coarse features interact.
    """
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     ← CHANGED: renamed to WhatNet-Kannada; num_classes=10; image_size=28.
#     All architectural decisions carry over unchanged.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_kannada(num_classes: int = 10, image_size: int = 28) -> Model:
    """
    WhatNet-Kannada: WhatNet adapted for Kannada-MNIST digit recognition.

    Architecture overview
    ─────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (texture / overall shape)
      • Cross-hair scaffold: 1×3 conv (horizontal) – captures short
        horizontal strokes and top/bottom curve segments of Kannada digits
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (14×14 at 28-px input)
      enc2: 64→128   ( 7× 7)
      enc3: 128→256  ( 4× 4, with padding)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – 1×3 + 3×1 + dilated branches
      • Gated fusion → final MLP + layer norm → logits (10 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    # ← CHANGED scaffold: 1×5 (Bengali headline) → 1×3 (Kannada horizontal strokes)
    s        = layers.Conv2D(32, (1, 3), padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(7)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    # ← CHANGED pool factor: 8 → 7  (28/4 = 7, not 4; keeps spatial dims aligned)
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    afc_in   = layers.GlobalAveragePooling2D()(enc3)
    afc_in   = layers.Add()([afc_in, cstb_out])
    afc_out  = adaptive_filter_capsule(afc_in, K)

    stgm_out    = stroke_topology_module(enc3, 256)
    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Kannada")  # ← CHANGED

MODELS_TF = {
    "WhatNet-Kannada": lambda: build_whatnet_kannada(NUM_CLASSES, IMG),  # ← CHANGED
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = -(-n_train // BS)   # ceiling division
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs  "
         f"[Kannada-MNIST]", "bold"))                                   # ← CHANGED
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "kannada_results.json")   # ← CHANGED
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kannada_histories.json")  # ← CHANGED
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kannada-MNIST benchmark complete.\n", "green", "bold"))  # ← CHANGED

[INFO] Loading train images from /kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST/X_kannada_MNIST_train.npz
[INFO] Loading train labels from /kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST/y_kannada_MNIST_train.npz
[INFO] Loading test images from /kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST/X_kannada_MNIST_test.npz
[INFO] Loading test labels from /kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST/y_kannada_MNIST_test.npz
[INFO] Classes in dataset : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[INFO] CFG num_classes    : 10
[INFO] Train: 54,000 | Val: 6,000 | Test: 10,000


I0000 00:00:1776578043.970653      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776578043.973574      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 50 epochs  [Kannada-MNIST]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Kannada  –  Parameter Summary                       ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d          ║  Conv2D               ║              288  ║
║  conv2d_1        ║  Conv2D               ║               96  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  dense           ║  Dense                ║              520  ║
║  dense_1         ║  Dense                ║              576  ║
║  conv2d_2        ║  Conv2D               ║            4,096  ║
║  batch_normal

I0000 00:00:1776578072.174979     167 service.cc:152] XLA service 0x7fc778019a40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776578072.175096     167 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776578072.175103     167 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776578077.084185     167 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776578101.798969     167 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Epoch   1/50  ░░░░░░░░░░░░░░░░░░░░   2.0%  loss=0.3927  acc=96.18%  val_loss=0.3499  val_acc=97.82%  lr=5.00e-04  [157.1s]
  Epoch   2/50  ░░░░░░░░░░░░░░░░░░░░   4.0%  loss=0.3115  acc=99.12%  val_loss=0.3867  val_acc=96.57%  lr=4.98e-04  [73.5s]
  Epoch   3/50  █░░░░░░░░░░░░░░░░░░░   6.0%  loss=0.3080  acc=99.21%  val_loss=0.3057  val_acc=99.22%  lr=4.96e-04  [75.9s]
  Epoch   4/50  █░░░░░░░░░░░░░░░░░░░   8.0%  loss=0.3035  acc=99.31%  val_loss=0.3109  val_acc=99.07%  lr=4.92e-04  [75.3s]
  Epoch   5/50  ██░░░░░░░░░░░░░░░░░░  10.0%  loss=0.3021  acc=99.38%  val_loss=0.3051  val_acc=99.23%  lr=4.88e-04  [75.9s]
  Epoch   6/50  ██░░░░░░░░░░░░░░░░░░  12.0%  loss=0.2998  acc=99.45%  val_loss=0.3068  val_acc=99.22%  lr=4.82e-04  [75.2s]
  Epoch   7/50  ██░░░░░░░░░░░░░░░░░░  14.0%  loss=0.2979  acc=99.49%  val_loss=0.3586  val_acc=97.42%  lr=4.76e-04  [75.4s]
  Epoch   8/50  ███░░░░░░░░░░░░░░░░░  16.0%  loss=0.2974  acc=99.53%  val_loss=0.3064  val_acc=99.20%  lr=4.69e-04  [75.3s]
  Epoch

# Method 3

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Kannada Handwritten Digit Recognition
#  Adapted from the Bengali (BanglaLekha-Isolated) version.
#
#  Transfer rationale
#  ──────────────────
#  Kannada is a South Indian Brahmic script.  Unlike Bengali/Devanagari it does
#  NOT have a shirorekha (top headline); instead characters are defined by
#  closed curves, loops, and prominent vertical/diagonal strokes.  Structural
#  changes vs. the Bengali version:
#
#    • num_classes   : 84  → 10   (Kannada-MNIST: digits 0–9 / ೦–೯)
#    • image_size    : 32  → 28   (Kannada-MNIST images are natively 28×28)
#    • Dataset format: folder-per-class (image files)
#                      → npz arrays  (Kannada-MNIST ships as .npz, identical
#                        layout to fashion-MNIST / MNIST)
#    • Dataset loader: Pillow file-walker → direct np.load() from .npz
#    • Scaffold branch: the 1×5 horizontal-headline conv is REPLACED by a
#                       1×3 + 3×1 cross-hair pair – better suited to the
#                       rounded, loop-heavy Kannada digit strokes.
#    • Decoder        : CSTB + STM + gated(STM, AFC)
#                       → Multi-scale GAP fusion + AFC + gated(dense, AFC)
#                         (mirrors the proven Devanagari decoder; lighter,
#                          no transformer overhead, no stroke-topology branch)
#    • STM / CSTB     : removed (no longer used in decoder)
#    • Model name     : WhatNet-BanglaLekha → WhatNet-Kannada
#    • Output files   : banglalekha_*.json  → kannada_*.json
#
#  Everything else – augmentation, LR schedule, callbacks, display utilities –
#  is unchanged.
#
#  Dataset
#  ───────
#  Kannada-MNIST  (Prabhu, 2019)
#    • 60 000 train + 10 000 test 28×28 greyscale images of Kannada digits
#    • Hosted on Kaggle:
#        https://www.kaggle.com/datasets/higgstachyon/kannada-mnist
#    • Download produces:
#        kannada_mnist_train.npz  –  keys: arr_0 (images), arr_1 (labels)
#        kannada_mnist_test.npz   –  keys: arr_0 (images), arr_1 (labels)
#      OR the CSV variants:
#        train.csv  /  test.csv   –  label in column 0, pixels in columns 1-784
#    • This script handles BOTH .npz and .csv automatically.
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Fields marked  ← CHANGED  differ from the Bengali version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 10 classes – Kannada digits ೦ ೧ ೨ ೩ ೪ ೫ ೬ ೭ ೮ ೯
    "num_classes":      10,
    # ← CHANGED: Kannada-MNIST images are natively 28×28 (same as MNIST)
    "image_size":       28,
    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,

    # Path to the Kannada-MNIST dataset root on Kaggle.
    #
    #   After adding the dataset to your notebook, the directory tree is:
    #
    #     /kaggle/input/kannada-mnist/
    #       Kannada_MNIST_npz/
    #         Dig_MNIST/          ← Dig-MNIST subset (not used here)
    #         Kannada_MNIST/      ← main split used by this script
    #           X_kannada_MNIST_train.npz   images  (60 000 × 28 × 28, uint8)
    #           y_kannada_MNIST_train.npz   labels  (60 000,           uint8)
    #           X_kannada_MNIST_test.npz    images  (10 000 × 28 × 28, uint8)
    #           y_kannada_MNIST_test.npz    labels  (10 000,           uint8)
    #
    #   Each .npz file contains a single array under the key 'arr_0'.
    #   Images are uint8 [0-255]; labels are uint8 [0-9].
    #
    #   Kaggle dataset page:
    #     https://www.kaggle.com/datasets/higgstachyon/kannada-mnist
    "data_dir":    "/kaggle/input/datasets/rautranjit/kannada-mnist",

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     Kannada-MNIST ships as split npz files (separate X / y per split) inside
#     a subdirectory.  Each .npz has a single array under key 'arr_0'.
#     Layout:
#       <data_dir>/Kannada_MNIST_npz/Kannada_MNIST/
#           X_kannada_MNIST_train.npz   y_kannada_MNIST_train.npz
#           X_kannada_MNIST_test.npz    y_kannada_MNIST_test.npz
# ─────────────────────────────────────────────────────────────────────────────

# Resolved subdirectory that holds the four .npz files
_NPZ_DIR = os.path.join(
    CFG["data_dir"], "Kannada_MNIST_npz", "Kannada_MNIST"
)

if not os.path.exists(CFG["data_dir"]):
    raise FileNotFoundError(
        f"[ERROR] Dataset root not found: {CFG['data_dir']}\n"
        "Add the dataset to your Kaggle notebook:\n"
        "  https://www.kaggle.com/datasets/higgstachyon/kannada-mnist\n"
        "Expected layout after mounting:\n"
        "  <data_dir>/Kannada_MNIST_npz/Kannada_MNIST/\n"
        "      X_kannada_MNIST_train.npz\n"
        "      y_kannada_MNIST_train.npz\n"
        "      X_kannada_MNIST_test.npz\n"
        "      y_kannada_MNIST_test.npz"
    )

if not os.path.exists(_NPZ_DIR):
    raise FileNotFoundError(
        f"[ERROR] Expected subdirectory not found: {_NPZ_DIR}\n"
        "The dataset appears to be mounted at the right root but the internal\n"
        "folder structure doesn't match.  Check that the dataset version on\n"
        "Kaggle still uses the path:\n"
        "  Kannada_MNIST_npz/Kannada_MNIST/<X|y>_kannada_MNIST_<train|test>.npz"
    )

def _load_split(split: str):
    """
    Load images and labels for one split from the two separate .npz files.
    Each file contains a single array under the key 'arr_0'.

    split: 'train'  →  60 000 samples (uint8 images, uint8 labels 0-9)
           'test'   →  10 000 samples (uint8 images, uint8 labels 0-9)
    """
    x_path = os.path.join(_NPZ_DIR, f"X_kannada_MNIST_{split}.npz")
    y_path = os.path.join(_NPZ_DIR, f"y_kannada_MNIST_{split}.npz")

    for p in (x_path, y_path):
        if not os.path.exists(p):
            raise FileNotFoundError(
                f"[ERROR] Missing file: {p}\n"
                f"Expected both X_kannada_MNIST_{split}.npz and "
                f"y_kannada_MNIST_{split}.npz in:\n  {_NPZ_DIR}"
            )

    print(f"[INFO] Loading {split} images from {x_path}")
    print(f"[INFO] Loading {split} labels from {y_path}")

    images = np.load(x_path)["arr_0"].astype(np.float32)  # (N, 28, 28)
    labels = np.load(y_path)["arr_0"].astype(np.int32)    # (N,)
    return images, labels

# ── Load raw arrays ───────────────────────────────────────────────────────────
x_train_all, y_train_all = _load_split("train")
x_test,      y_test      = _load_split("test")

print(f"[INFO] Classes in dataset : {np.unique(y_train_all).tolist()}")
print(f"[INFO] CFG num_classes    : {NUM_CLASSES}")
if len(np.unique(y_train_all)) != NUM_CLASSES:
    print(f"[WARN] Mismatch – update CFG['num_classes'] to "
          f"{len(np.unique(y_train_all))} to match your dataset.")

# ── Train / val split ─────────────────────────────────────────────────────────
n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

# Shuffle before splitting (dataset is class-sorted in the .npz)
rng  = np.random.default_rng(SEED)
perm = rng.permutation(n_total)
x_train_all, y_train_all = x_train_all[perm], y_train_all[perm]

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

# ── Add channel dim: (N,28,28) → (N,28,28,1) ─────────────────────────────────
x_train = x_train[..., np.newaxis]
x_val   = x_val[...,   np.newaxis]
x_test  = x_test[...,  np.newaxis]

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Light stochastic augmentation – training only.
    Kannada digit glyphs are compact and mostly upright, so we keep
    augmentations mild: brightness/contrast jitter + pad-and-random-crop
    for random translation.  No rotation – it would corrupt loop topology.
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
#    Images are already in RAM as numpy arrays, so from_tensor_slices is
#    ideal – no file I/O or Pillow decoding required.
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
# test_ds      – raw integer labels  (used by compute_macro_f1)
# test_ds_oh   – one-hot labels      (used by model.evaluate)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    """GELU activation – smoother than ReLU, better gradients in deep nets."""
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    """Standard pre-activation residual block."""
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """
    DenseNet-inspired residual block.
    Three sequential residual blocks with dense connections, then a
    1×1 bottleneck and stride-2 depthwise downsampling.
    """
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    """Squeeze-and-Excitation (SE) channel attention."""
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    """
    Lightweight capsule-like routing module.
    Projects the feature vector into a (num_classes × capsule_dim) tensor,
    then uses the original feature as a per-class filter and sums to produce
    class-discriminative logit-like scores.  No dynamic routing – O(n) cost.
    """
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

# ─────────────────────────────────────────────────────────────────────────────
#  NOTE: stroke_topology_module and cross_scale_transformer_bridge are no
#  longer used in the decoder.  They are removed to keep the file clean.
#  The decoder now mirrors the Devanagari version exactly:
#    multi-scale GAP fusion → AFC → gated(dense head, AFC) → logits
# ─────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     ← CHANGED decoder: CSTB + STM + gated(STM, AFC)
#                       → multi-scale GAP + AFC + gated(dense, AFC)
#       (mirrors proven Devanagari decoder; lighter, no transformer overhead)
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_kannada(
    num_classes: int = 10,
    image_size:  int = 28,
    head_units:  int = 256,
) -> Model:
    """
    WhatNet-Kannada: WhatNet adapted for Kannada-MNIST digit recognition.

    Architecture overview
    ─────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (texture / overall shape)
      • Cross-hair scaffold: 1×3 conv (horizontal) – captures short
        horizontal strokes and top/bottom curve segments of Kannada digits
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (14×14 at 28-px input)
      enc2: 64→128   ( 7× 7)
      enc3: 128→256  ( 4× 4, with padding)
      Weighted scaffold residuals injected at each stage.

    Decoder head  ← CHANGED to match Devanagari version:
      • Multi-scale GAP fusion: enc1 + enc2 + enc3 pooled and concatenated
      • Adaptive Filter Capsule (AFC): class-discriminative capsule scores
      • Dense head: direct linear projection of the fused GAP vector
      • Gated fusion: per-sample softmax gate blends dense-head logits and
        AFC scores → final logits (10 classes)

    Removed vs. previous Kannada version:
      • Cross-scale transformer bridge (CSTB)  – no longer needed
      • Stroke topology module (STM)           – replaced by multi-scale GAP
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    # ← CHANGED scaffold: 1×5 (Bengali headline) → 1×3 (Kannada horizontal
    #   strokes; no shirorekha in Kannada)
    s        = layers.Conv2D(32, (1, 3), padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(7)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    # ← pool factor 7 = 28/4; keeps spatial dims aligned at this resolution
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head  ← CHANGED: mirrors Devanagari GAP+AFC+gated fusion ──
    #
    # Step 1 – Multi-scale GAP fusion
    #   Pool all three encoder stages to vectors and concatenate.
    #   This gives the classifier access to fine (enc1) and coarse (enc3)
    #   features without the overhead of a transformer bridge.
    gap1      = layers.GlobalAveragePooling2D(name="gap1")(enc1)   # (B, 64)
    gap2      = layers.GlobalAveragePooling2D(name="gap2")(enc2)   # (B, 128)
    gap3      = layers.GlobalAveragePooling2D(name="gap3")(enc3)   # (B, 256)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])
    # fused_gap shape: (B, 448)

    # Step 2 – Adaptive Filter Capsule (AFC)
    #   Projects the fused multi-scale vector into capsule space.
    #   Each of the K capsules learns to respond to one Kannada digit class.
    afc_out = adaptive_filter_capsule(fused_gap, K)                # (B, K)

    # Step 3 – Dense classification head (residual path alongside AFC)
    x = layers.Dense(head_units, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation(gelu, name="head_act")(x)
    x = layers.Dense(K, name="head_logits")(x)                    # (B, K)

    # Step 4 – Gated fusion
    #   A learned per-sample gate (softmax over 2 weights) blends the dense
    #   head logits with the AFC capsule scores.  This lets the model learn
    #   how much to trust direct projection vs. capsule routing.
    #   gate[:,0] weights the dense head; gate[:,1] weights the AFC output.
    combined   = layers.Concatenate(name="gate_input")([x, afc_out])
    gate       = layers.Dense(2, activation="softmax", name="gate")(combined)
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x,       gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc"  )([afc_out, gate])
    outputs    = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=outputs, name="WhatNet-Kannada")


MODELS_TF = {
    "WhatNet-Kannada": lambda: build_whatnet_kannada(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = -(-n_train // BS)   # ceiling division
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs  "
         f"[Kannada-MNIST]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "kannada_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kannada_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kannada-MNIST benchmark complete.\n", "green", "bold"))

2026-04-24 14:13:08.411394: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777039988.634636      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777039988.703551      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777039989.233409      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777039989.233445      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777039989.233448      55 computation_placer.cc:177] computation placer alr

[INFO] Loading train images from /kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST/X_kannada_MNIST_train.npz
[INFO] Loading train labels from /kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST/y_kannada_MNIST_train.npz
[INFO] Loading test images from /kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST/X_kannada_MNIST_test.npz
[INFO] Loading test labels from /kaggle/input/datasets/rautranjit/kannada-mnist/Kannada_MNIST_npz/Kannada_MNIST/y_kannada_MNIST_test.npz
[INFO] Classes in dataset : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[INFO] CFG num_classes    : 10
[INFO] Train: 54,000 | Val: 6,000 | Test: 10,000


I0000 00:00:1777040016.900537      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777040016.906639      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 100 epochs  [Kannada-MNIST]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Kannada  –  Parameter Summary                       ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d          ║  Conv2D               ║              288  ║
║  conv2d_1        ║  Conv2D               ║               96  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  dense           ║  Dense                ║              520  ║
║  dense_1         ║  Dense                ║              576  ║
║  conv2d_2        ║  Conv2D               ║            4,096  ║
║  batch_norma

I0000 00:00:1777040040.560024     127 service.cc:152] XLA service 0x7f45c8004350 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777040040.560063     127 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777040040.560067     127 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777040044.526265     127 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777040064.311457     127 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Epoch   1/100  ░░░░░░░░░░░░░░░░░░░░   1.0%  loss=0.6684  acc=96.55%  val_loss=0.5588  val_acc=98.35%  lr=5.00e-04  [142.0s]
  Epoch   2/100  ░░░░░░░░░░░░░░░░░░░░   2.0%  loss=0.5451  acc=99.17%  val_loss=0.5410  val_acc=98.88%  lr=5.00e-04  [73.1s]
  Epoch   3/100  ░░░░░░░░░░░░░░░░░░░░   3.0%  loss=0.5303  acc=99.35%  val_loss=0.5203  val_acc=99.35%  lr=4.99e-04  [71.9s]
  Epoch   4/100  ░░░░░░░░░░░░░░░░░░░░   4.0%  loss=0.5262  acc=99.40%  val_loss=0.5158  val_acc=99.57%  lr=4.98e-04  [71.9s]
  Epoch   5/100  █░░░░░░░░░░░░░░░░░░░   5.0%  loss=0.5242  acc=99.44%  val_loss=0.5252  val_acc=99.35%  lr=4.97e-04  [71.1s]
  Epoch   6/100  █░░░░░░░░░░░░░░░░░░░   6.0%  loss=0.5210  acc=99.49%  val_loss=0.5230  val_acc=99.37%  lr=4.96e-04  [71.2s]
  Epoch   7/100  █░░░░░░░░░░░░░░░░░░░   7.0%  loss=0.5192  acc=99.52%  val_loss=0.5165  val_acc=99.58%  lr=4.94e-04  [71.8s]
  Epoch   8/100  █░░░░░░░░░░░░░░░░░░░   8.0%  loss=0.5178  acc=99.59%  val_loss=0.5273  val_acc=99.52%  lr=4.92e-04  [71.2s]